pip install pyvisgraph

In [1]:
import pyvisgraph as vg
polys = [[vg.Point(0.0,1.0), vg.Point(3.0,1.0), vg.Point(1.5,4.0)],
         [vg.Point(4.0,4.0), vg.Point(7.0,4.0), vg.Point(5.5,8.0)]]
g = vg.VisGraph()
g.build(polys)
shortest = g.shortest_path(vg.Point(1.5,0.0), vg.Point(4.0, 6.0))
print(shortest)

100%|██████████| 1/1 [00:00<00:00, 999.36it/s]

[Point(1.50, 0.00), Point(3.00, 1.00), Point(4.00, 6.00)]


1. build_graph_from_shapefiles

In [3]:
import pyvisgraph as vg
import shapefile

# In this example a visibility graph will be created from the GSHHS shorelines
# available at: https://www.ngdc.noaa.gov/mgg/shorelines/gshhs.html
# 1. Download the shorelines as shape files and select the wanted resolution 
#    and level (in this example we use Crude, L1). 
# 2. Copy all the related files (shx, shp, prj, dbf) to the folder that contains
#    this script.

input_shapefile = shapefile.Reader('GSHHS_c_L1')
output_graphfile = 'GSHHS_c_L1.graph'
# Number of CPU cores on host computer. If you don't know how many cores you
# have, use 'cat /proc/cpuinfo | grep processor | wc -l' on Linux. On Windows,
# press Ctrl + Shift + Esc, press Performance tab. Look for 'logical processors'.
workers = 4 

# Get the shoreline shapes from the shape file. Broadly speaking the GSHHS
# shapes correspond to the shorelines of continents, countries and islands.
shapes = input_shapefile.shapes()
print('The shapefile contains {} shapes.'.format(len(shapes)))

# Create a list of polygons, where each polygon corresponds to a shape
polygons = []
for shape in shapes:
    polygon = []
    for point in shape.points:
        polygon.append(vg.Point(point[0], point[1]))
    polygons.append(polygon)

# Start building the visibility graph 
graph = vg.VisGraph()
print('Starting building visibility graph')
graph.build(polygons, workers=workers)
print('Finished building visibility graph')

# Save the visibility graph to a file 
graph.save(output_graphfile)
print('Saved visibility graph to file: {}'.format(output_graphfile))

ModuleNotFoundError: No module named 'shapefile'

In [2]:
import pyvisgraph as vg
import folium

# In this example we will calculate the shortest path between two points
# and plot this on a interactive map, using the folium package.

# Example points
start_point = vg.Point(12.568337, 55.676098) # Copenhagen
end_point = vg.Point(103.851959, 1.290270) # Singapore

# Load the visibility graph file If you do not have this, please run
# 1_build_graph_from_shapefiles.py first.
graph = vg.VisGraph()
graph.load('GSHHS_c_L1.graph')

# Calculate the shortest path
shortest_path  = graph.shortest_path(start_point, end_point)

# Plot of the path using folium
geopath = [[point.y, point.x] for point in shortest_path]
geomap  = folium.Map([0, 0], zoom_start=2)
for point in geopath:
    folium.Marker(point, popup=str(point)).add_to(geomap)
folium.PolyLine(geopath).add_to(geomap)

# Add a Mark on the start and positions in a different color
folium.Marker(geopath[0], popup=str(start_point), icon=folium.Icon(color='red')).add_to(geomap)
folium.Marker(geopath[-1], popup=str(end_point), icon=folium.Icon(color='red')).add_to(geomap)

# Save the interactive plot as a map
output_name = 'example_shortest_path_plot.html'
geomap.save(output_name)
print('Output saved to: {}'.format(output_name))

FileNotFoundError: [Errno 2] No such file or directory: 'GSHHS_c_L1.graph'